# CS-781 Phase 7 — Architecture A Long-Run Legal Retrieval (resumable)

**Dedicated Architecture-A full-fine-tuning experiment.** This is an additional PCR experiment, not a replacement for the main Phase 7 requirement of six distinct techniques/submissions.

## Method
- Phase-1 **LegalDecoderA**, using the final `recommended_500m/best_model.pt` when attached and verified. Reported Phase-1 size: 243,897,344 parameters; 18 decoder blocks; width 1,024; 16 query heads / 4 KV heads; 2,048 context; RoPE, RMSNorm and SwiGLU.
- Full fine-tuning with a learned retrieval projection over the last-token and masked-mean representations.
- Three deterministic legal-text views (opening, middle and ending) per case.
- TF-IDF hard negatives mined only from the official Phase-7 training candidate pool, plus random in-pool negatives.
- Contrastive softmax + pairwise ranking objective.
- Periodic full-development Micro-F1@7 evaluation, best-checkpoint selection, then final dev-tuned neural/lexical score fusion.
- Test retrieval only from `test_candidates.json`; exactly seven unique ranked IDs per test query; final ZIP has only root-level `prediction.json`.

## Long-run / resume plan
The target is **13.5 hours of optimizer time**, with approximately **15–18 hours end-to-end** including token-cache construction, hard-negative mining, validation, and final retrieval. Kaggle documents a 12-hour maximum per CPU/GPU notebook session and 20 GB of saved `/kaggle/working` output, so the notebook stops before the 12-hour cap and resumes in a second session. See Kaggle's [Notebook documentation](https://www.kaggle.com/docs/notebooks).

At each checkpoint it saves FP32 model weights, AdamW state, scheduler, GradScaler, optimizer step, accumulated training time, and Python/NumPy/PyTorch/CUDA RNG state. Checkpoints are written to a temporary file and atomically replaced. The notebook also saves token caches, TF-IDF artifacts, hard-negative lists, periodic best-dev model weights, and test-embedding progress.

**Resume:** at the safe stop, use **Save Version → Quick Save**. In the next session, use **Add Input → Notebook Output** to attach the previous version, then rerun all cells. The notebook discovers the previous run manifest, restores the run directory, checks the exact checkpoint/tokenizer/dataset fingerprints, and continues from the saved optimizer step. Do not delete the previous output.

The uploaded multi-architecture run you shared did not create valid submissions: E1–E5 trained for 140 steps but failed when writing `dev_predictions.json` because their per-experiment output directories were missing; E6 was blocked because E1 had not saved a teacher. This Architecture-A notebook creates its output directories before every write and runs independently from the Phase-1 A checkpoint.

In [ ]:
import os, sys, json, re, gc, math, time, random, hashlib, shutil, zipfile, warnings, traceback, copy
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

SEED = 781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('Architecture-A long-run retrieval requires a Kaggle GPU. Select GPU (T4 or T4 x2).')
DEVICE = torch.device('cuda:0')
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print('GPUs:', GPU_NAMES, '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)

# Kaggle supports up to 12h per CPU/GPU session. Stop early to leave time for checkpoint writes.
SESSION_SAFE_HOURS = 10.20
TOTAL_TRAIN_HOURS = 13.50
SESSION_STOP_RESERVE_MIN = 30
CHECKPOINT_EVERY_STEPS = 100
LOG_EVERY_STEPS = 25
EVAL_EVERY_TRAIN_HOURS = 2.50
GROUP_BATCH_SIZE = 1              # one query + one positive + two negatives = four 256-token sequences
NEGATIVES_PER_QUERY = 2
DOC_CHUNKS = 3                    # opening / middle / ending
DOC_CHUNK_TOKENS = 256
TRAIN_LR = 1.5e-5
WEIGHT_DECAY = 0.01
TEMPERATURE = 0.07
PAIRWISE_MARGIN = 0.15
MAX_TRAIN_STEPS = 100_000
EVAL_BATCH_SEQUENCES = 8          # inference-only batches of token chunks
EMBED_BATCH_DOCS = 4              # four documents = at most 12 chunks per inference batch
MIN_FREE_GB_FOR_RESUME_SAVE = 7.0 # old checkpoint + new temporary checkpoint + safety margin

ROOT = Path('/kaggle/working/CS781_PHASE7_ARCHA_LONGRUN')
CACHE_ROOT = ROOT/'token_cache'
EXP_DIR = ROOT/'experiments'/'A_FullFineTune_LongRun'
SUB_DIR = ROOT/'submissions'
for p in (ROOT, CACHE_ROOT, EXP_DIR, SUB_DIR): p.mkdir(parents=True, exist_ok=True)
SESSION_START = time.time()
SESSION_DEADLINE = SESSION_START + SESSION_SAFE_HOURS*3600
TRAIN_BUDGET_SECONDS = TOTAL_TRAIN_HOURS*3600
RUN_MAGIC = 'CS781_PHASE7_ARCHA_LONGRUN_V2'
RUN_MANIFEST = ROOT/'run_manifest.json'

# Atomic file writers. Temporary files are ignored when an earlier Notebook Output is restored.
def atomic_json(obj, path):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    os.replace(tmp, path)

def atomic_torch_save(obj, path):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(obj, tmp)
    os.replace(tmp, path)

def free_gb(path='/kaggle/working'):
    return shutil.disk_usage(path).free/(1024**3)

def _restore_prior_output():
    local_latest=EXP_DIR/'latest_resume.pt'
    if local_latest.is_file():
        print('Found local Architecture-A resume checkpoint; retaining it.')
        return None
    found=[]
    inp=Path('/kaggle/input')
    if inp.exists():
        for manifest_path in inp.rglob('run_manifest.json'):
            try:
                meta=json.loads(manifest_path.read_text(encoding='utf-8'))
                source=manifest_path.parent
                source_cp=source/'experiments'/'A_FullFineTune_LongRun'/'latest_resume.pt'
                if meta.get('magic')==RUN_MAGIC and source_cp.is_file():
                    found.append((source_cp.stat().st_mtime, source, source_cp))
            except Exception:
                continue
    if not found:
        print('No previous Architecture-A output attached; starting a new run.')
        return None
    _, source, source_cp = sorted(found, key=lambda x:x[0], reverse=True)[0]
    print('Restoring prior Notebook Output:', source, '| checkpoint GiB:', round(source_cp.stat().st_size/(1024**3),2))
    shutil.copytree(source, ROOT, dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.tmp','*.partial','*.build.npy'))
    for p in (ROOT,CACHE_ROOT,EXP_DIR,SUB_DIR): p.mkdir(parents=True,exist_ok=True)
    if not (EXP_DIR/'latest_resume.pt').is_file():
        raise RuntimeError('Previous output copy completed but latest_resume.pt is missing. Do not start training from an incomplete output.')
    return source

RESTORED_FROM = _restore_prior_output()
if not RUN_MANIFEST.exists():
    atomic_json({'magic':RUN_MAGIC,'created':time.strftime('%Y-%m-%d %H:%M:%S'),
        'architecture':'A / LegalDecoderA','method':'full_fine_tuning','training_budget_hours':TOTAL_TRAIN_HOURS},RUN_MANIFEST)
print('Run root:',ROOT)
print(f'Session safe budget: {SESSION_SAFE_HOURS:.2f}h | optimizer-time budget: {TOTAL_TRAIN_HOURS:.2f}h')
print(f'Free working disk: {free_gb():.2f} GiB / 20 GiB Kaggle saved-output limit')


In [ ]:

def get_hf_token():
    tok=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if tok: return tok.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        cli=UserSecretsClient()
        for name in ('HF','HF_TOKEN'):
            try:
                val=cli.get_secret(name)
                if val and val.strip(): return val.strip()
            except Exception: pass
    except Exception: pass
    return None

HF_TOKEN=get_hf_token()
REPO='Exploration-Lab/CS-781-Capstone'
DATA_ROOT=None
for base in (Path('/kaggle/input'),Path('/kaggle/working'),Path('/root/.cache/huggingface/hub')):
    hits=list(base.glob('**/Phase_7_PCR/Train/train.json')) if base.exists() else []
    if hits:
        DATA_ROOT=hits[0].parents[1]; break
if DATA_ROOT is None:
    from huggingface_hub import snapshot_download
    snap=Path(snapshot_download(repo_id=REPO,repo_type='dataset',token=HF_TOKEN,allow_patterns=['Phase_7_PCR/**']))
    if (snap/'Phase_7_PCR').exists(): DATA_ROOT=snap/'Phase_7_PCR'
if DATA_ROOT is None: raise FileNotFoundError('Phase_7_PCR data not available. Enable Internet and check HF access.')
EXPECTED=['Train/train.json','Train/train_candidates.json','Dev/dev.json','Dev/dev_candidates.json','Test/test.json','Test/test_candidates.json']
for rel in EXPECTED:
    if not (DATA_ROOT/rel).is_file(): raise FileNotFoundError(f'Missing official Phase-7 file: {DATA_ROOT/rel}')
print('Official Phase-7 root:',DATA_ROOT,'| all six required split files present')

TOKENIZER_SHA256='3e38dd5c561d84ad8fbadf792bbca490f8c70346d922df7afbef37c5a942b71e'
def iter_files(base,suffixes=None,names=None):
    base=Path(base)
    if not base.exists(): return
    skip={'.git','__pycache__','.cache','node_modules'}
    for root,dirs,files in os.walk(base):
        dirs[:]=[d for d in dirs if d not in skip]
        for fn in files:
            p=Path(root)/fn
            if suffixes is not None and p.suffix.lower() not in suffixes: continue
            if names is not None and fn not in names: continue
            yield p

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

tokenizer_path=None
for base in (Path('/kaggle/input'),Path('/kaggle/working')):
    for p in iter_files(base,names={'tokenizer.json'}):
        try:
            if sha256_file(p)==TOKENIZER_SHA256: tokenizer_path=p; break
        except Exception: pass
    if tokenizer_path: break
if tokenizer_path is None: raise FileNotFoundError('Exact LEGAL_40K tokenizer not found. Attach the Phase-1 training-output dataset.')
from tokenizers import Tokenizer
TOKENIZER=Tokenizer.from_file(str(tokenizer_path))
assert TOKENIZER.get_vocab_size()==40000
print('Verified LEGAL_40K tokenizer:',tokenizer_path,'| SHA256:',TOKENIZER_SHA256)

# Prefer Architecture A's final recommended_500m best checkpoint, then use the research best as fallback.
CHECKPOINT_PATHS=[
    Path('/kaggle/input/datasets/rohith00000077/arch-a-500m-training-continue/CS781_PHASE1_MASTER/checkpoints/final/recommended_500m/best_model.pt'),
    Path('/kaggle/input/datasets/rohith00000077/arch-a/CS781_PHASE1_RESEARCH/training/checkpoints_archA_large/best_model.pt'),
    Path('/kaggle/input/datasets/rohith00000077/arch-a/CS781_PHASE1_RESEARCH/training/checkpoints_archA_large/latest.pt'),
]
for base in (Path('/kaggle/input'),Path('/kaggle/working')):
    for p in iter_files(base,suffixes={'.pt','.pth','.ckpt'}):
        if p.name.lower()=='best_model.pt' and ('arch-a' in str(p).lower() or 'architecturea' in str(p).lower()):
            CHECKPOINT_PATHS.append(p)
seen=set(); CHECKPOINT_PATHS=[p for p in CHECKPOINT_PATHS if p.is_file() and not (str(p) in seen or seen.add(str(p)))]
if not CHECKPOINT_PATHS: raise FileNotFoundError('Architecture-A checkpoint not found. Attach arch-a-500m-training-continue or arch-a Phase-1 inputs.')

def extract_cfg(payload):
    for key in ('model_config','MODEL_CONFIG','config','cfg','model_cfg'):
        val=payload.get(key) if isinstance(payload,dict) else None
        if isinstance(val,dict) and 'vocab_size' in val and 'd_model' in val: return dict(val)
    return None

def extract_state(payload):
    for key in ('model_state_dict','state_dict','model_state','weights','model'):
        val=payload.get(key) if isinstance(payload,dict) else None
        if isinstance(val,dict) and val and any(torch.is_tensor(x) for x in val.values()): return val
    if isinstance(payload,dict) and payload and all(torch.is_tensor(v) for v in payload.values()): return payload
    return None

checkpoint_reports=[]
for p in CHECKPOINT_PATHS:
    pl=None
    try:
        try: pl=torch.load(p,map_location='cpu',weights_only=False,mmap=True)
        except Exception: pl=torch.load(p,map_location='cpu',weights_only=False)
        cfg=extract_cfg(pl); st=extract_state(pl)
        if cfg and st:
            params=sum(v.numel() for v in st.values() if torch.is_tensor(v))
            checkpoint_reports.append({'path':str(p),'cfg':cfg,'step':pl.get('step',pl.get('global_step',-1)),
                'val_loss':pl.get('validation_loss',pl.get('best_val_loss',float('inf'))),'state_tensors':len(st),'state_parameter_entries':params})
    except Exception as exc: print('Checkpoint inspection warning:',p,repr(exc)[:200])
    finally:
        if pl is not None: del pl
        gc.collect()
if not checkpoint_reports: raise RuntimeError('No compatible Architecture-A checkpoint metadata/state found.')
checkpoint_reports.sort(key=lambda x:(0 if 'arch-a-500m-training-continue' in x['path'] else 1,
                                      0 if Path(x['path']).name=='best_model.pt' else 1,
                                      float(x['val_loss']) if isinstance(x['val_loss'],(int,float)) else float('inf')))
BASE_CHECKPOINT=Path(checkpoint_reports[0]['path'])
MODEL_CONFIG=checkpoint_reports[0]['cfg']
BASE_CHECKPOINT_SHA256=sha256_file(BASE_CHECKPOINT)
print('Selected Phase-1 A checkpoint:',BASE_CHECKPOINT)
print('Checkpoint SHA256:',BASE_CHECKPOINT_SHA256)
print('Checkpoint metadata:',json.dumps({k:v for k,v in checkpoint_reports[0].items() if k!='cfg'},indent=2))
print('Model config:',json.dumps(MODEL_CONFIG,indent=2))
assert int(MODEL_CONFIG['vocab_size'])==40000
assert int(MODEL_CONFIG['d_model'])==1024 and int(MODEL_CONFIG['n_layers'])==18
assert int(MODEL_CONFIG['context_length'])>=512
print('Phase-1 Architecture-A configuration checks passed.')


In [ ]:


class RMSNorm(nn.Module):

    def __init__(
        self,
        dim,
        eps=1e-5
    ):

        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        inv_rms = torch.rsqrt(
            x.pow(2).mean(
                dim=-1,
                keepdim=True
            )
            + self.eps
        )

        return (
            x
            * inv_rms
            * self.weight
        )


class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        dim,
        max_seq_len,
        theta=10000.0
    ):

        super().__init__()

        if dim % 2:
            raise ValueError(
                "RoPE dimension must be even."
            )

        inv_freq = 1.0 / (
            theta ** (
                torch.arange(
                    0,
                    dim,
                    2,
                    dtype=torch.float32
                )
                / dim
            )
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32
        )

        freqs = torch.outer(
            positions,
            inv_freq
        )

        self.register_buffer(
            "cos_cached",
            torch.repeat_interleave(
                freqs.cos(),
                2,
                dim=-1
            ),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            torch.repeat_interleave(
                freqs.sin(),
                2,
                dim=-1
            ),
            persistent=False
        )

    def forward(
        self,
        x,
        position_offset=0
    ):

        seq_len = x.size(-2)

        end = (
            position_offset
            + seq_len
        )

        if (
            end
            > self.cos_cached.size(0)
        ):
            raise ValueError(
                "RoPE context exceeded."
            )

        cos = self.cos_cached[
            position_offset:end
        ].to(
            device=x.device,
            dtype=x.dtype
        )

        sin = self.sin_cached[
            position_offset:end
        ].to(
            device=x.device,
            dtype=x.dtype
        )

        cos = cos.view(
            1,
            1,
            seq_len,
            -1
        )

        sin = sin.view(
            1,
            1,
            seq_len,
            -1
        )

        even = x[..., 0::2]
        odd = x[..., 1::2]

        rotated = torch.stack(
            [
                -odd,
                even
            ],
            dim=-1
        ).flatten(-2)

        return (
            x * cos
            + rotated * sin
        )


class GQAAttention(nn.Module):

    def __init__(
        self,
        cfg
    ):

        super().__init__()

        d_model = cfg[
            "d_model"
        ]

        n_heads = cfg[
            "n_heads"
        ]

        n_kv_heads = cfg[
            "n_kv_heads"
        ]

        head_dim = (
            d_model // n_heads
        )

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim

        self.group_size = (
            n_heads
            // n_kv_heads
        )

        self.q_proj = nn.Linear(
            d_model,
            n_heads * head_dim,
            bias=cfg["use_bias"]
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * head_dim,
            bias=cfg["use_bias"]
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * head_dim,
            bias=cfg["use_bias"]
        )

        self.o_proj = nn.Linear(
            d_model,
            d_model,
            bias=cfg["use_bias"]
        )

        self.rope = RotaryEmbedding(
            head_dim,
            cfg["context_length"],
            cfg["rope_theta"]
        )

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False
    ):

        batch, seq_len, d_model = (
            x.shape
        )

        past_len = (
            0
            if past_kv is None
            else past_kv[0].size(2)
        )

        q = self.q_proj(
            x
        ).view(
            batch,
            seq_len,
            self.n_heads,
            self.head_dim
        ).transpose(
            1,
            2
        )

        k = self.k_proj(
            x
        ).view(
            batch,
            seq_len,
            self.n_kv_heads,
            self.head_dim
        ).transpose(
            1,
            2
        )

        v = self.v_proj(
            x
        ).view(
            batch,
            seq_len,
            self.n_kv_heads,
            self.head_dim
        ).transpose(
            1,
            2
        )

        q = self.rope(
            q,
            past_len
        )

        k = self.rope(
            k,
            past_len
        )

        if past_kv is not None:

            past_k, past_v = past_kv

            k = torch.cat(
                [past_k, k],
                dim=2
            )

            v = torch.cat(
                [past_v, v],
                dim=2
            )

        if (
            self.n_kv_heads
            != self.n_heads
        ):

            k_attn = (
                k.repeat_interleave(
                    self.group_size,
                    dim=1
                )
            )

            v_attn = (
                v.repeat_interleave(
                    self.group_size,
                    dim=1
                )
            )

        else:

            k_attn = k
            v_attn = v

        if past_kv is None:

            y = F.scaled_dot_product_attention(
                q,
                k_attn,
                v_attn,
                attn_mask=None,
                dropout_p=0.0,
                is_causal=True,
            )

        else:

            y = F.scaled_dot_product_attention(
                q,
                k_attn,
                v_attn,
                attn_mask=None,
                dropout_p=0.0,
                is_causal=False,
            )

        y = y.transpose(
            1,
            2
        ).contiguous().view(
            batch,
            seq_len,
            d_model
        )

        output = self.o_proj(
            y
        )

        if use_cache:
            return output, (k, v)

        return output


class SwiGLU(nn.Module):

    def __init__(
        self,
        cfg
    ):

        super().__init__()

        d_model = cfg[
            "d_model"
        ]

        ffn_dim = cfg[
            "ffn_dim"
        ]

        self.gate_proj = nn.Linear(
            d_model,
            ffn_dim,
            bias=cfg["use_bias"]
        )

        self.up_proj = nn.Linear(
            d_model,
            ffn_dim,
            bias=cfg["use_bias"]
        )

        self.down_proj = nn.Linear(
            ffn_dim,
            d_model,
            bias=cfg["use_bias"]
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(
                self.gate_proj(x)
            )
            * self.up_proj(x)
        )


class DecoderBlock(nn.Module):

    def __init__(
        self,
        cfg
    ):

        super().__init__()

        d_model = cfg[
            "d_model"
        ]

        self.norm1 = RMSNorm(
            d_model,
            cfg["norm_eps"]
        )

        self.attn = GQAAttention(
            cfg
        )

        self.norm2 = RMSNorm(
            d_model,
            cfg["norm_eps"]
        )

        self.ffn = SwiGLU(
            cfg
        )

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False
    ):

        if use_cache:

            attn_out, present = (
                self.attn(
                    self.norm1(x),
                    past_kv=past_kv,
                    use_cache=True,
                )
            )

            x = x + attn_out

            x = x + self.ffn(
                self.norm2(x)
            )

            return x, present

        x = x + self.attn(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


class LegalDecoderA(nn.Module):

    def __init__(
        self,
        cfg
    ):

        super().__init__()

        self.cfg = dict(cfg)

        self.token_embedding = nn.Embedding(
            cfg["vocab_size"],
            cfg["d_model"]
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(cfg)
            for _ in range(
                cfg["n_layers"]
            )
        ])

        self.final_norm = RMSNorm(
            cfg["d_model"],
            cfg["norm_eps"]
        )

        self.lm_head = nn.Linear(
            cfg["d_model"],
            cfg["vocab_size"],
            bias=False
        )

        if cfg["tie_embeddings"]:

            self.lm_head.weight = (
                self.token_embedding.weight
            )

        self.apply(
            self._init_weights
        )

        residual_scale = (
            1.0
            / math.sqrt(
                2.0
                * cfg["n_layers"]
            )
        )

        for block in self.blocks:

            nn.init.normal_(
                block.attn.o_proj.weight,
                mean=0.0,
                std=0.02
                * residual_scale,
            )

            nn.init.normal_(
                block.ffn.down_proj.weight,
                mean=0.0,
                std=0.02
                * residual_scale,
            )

    def _init_weights(
        self,
        module
    ):

        if isinstance(
            module,
            nn.Linear
        ):

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

        elif isinstance(
            module,
            nn.Embedding
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

        elif isinstance(
            module,
            RMSNorm
        ):

            nn.init.ones_(
                module.weight
            )

    def forward(
        self,
        input_ids,
        labels=None,
        use_cache=False,
        past_key_values=None
    ):

        x = self.token_embedding(
            input_ids
        )

        if use_cache:

            if past_key_values is None:

                past_key_values = [
                    None
                    for _ in self.blocks
                ]

            presents = []

            for block, past in zip(
                self.blocks,
                past_key_values
            ):

                x, present = block(
                    x,
                    past_kv=past,
                    use_cache=True
                )

                presents.append(
                    present
                )

            x = self.final_norm(
                x
            )

            logits = self.lm_head(
                x
            )

            return {
                "logits": logits,
                "loss": None,
                "past_key_values": presents,
            }

        for block in self.blocks:

            if (
                self.training
                and self.cfg[
                    "gradient_checkpointing"
                ]
            ):

                x = torch.utils.checkpoint.checkpoint(
                    block,
                    x,
                    use_reentrant=False
                )

            else:

                x = block(x)

        x = self.final_norm(
            x
        )

        logits = self.lm_head(
            x
        )

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                labels.reshape(
                    -1
                )
            )

        return {
            "logits": logits,
            "loss": loss,
        }


print(
    "Architecture A model classes: READY"
)

In [ ]:

class RetrievalA(nn.Module):
    """Phase-1 LegalDecoderA backbone plus a learned retrieval projection."""
    def __init__(self,backbone,dim=1024,dropout=0.10):
        super().__init__()
        self.backbone=backbone
        self.projection=nn.Sequential(
            nn.Linear(2*dim,dim,bias=True),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(dim),
        )
    def forward(self,input_ids,lengths):
        if input_ids.ndim!=2 or lengths.ndim!=1 or input_ids.size(0)!=lengths.numel():
            raise ValueError('Expected input_ids [batch, sequence] and lengths [batch].')
        if torch.any(lengths<1) or torch.any(lengths>input_ids.size(1)):
            raise ValueError('Invalid document token lengths.')
        x=self.backbone.token_embedding(input_ids)
        for block in self.backbone.blocks:
            if self.training and bool(self.backbone.cfg.get('gradient_checkpointing',True)):
                x=checkpoint(block,x,use_reentrant=False)
            else:
                x=block(x)
        x=self.backbone.final_norm(x)
        rows=torch.arange(x.size(0),device=x.device)
        last=x[rows,(lengths-1).clamp_min(0)].float()
        positions=torch.arange(x.size(1),device=x.device).unsqueeze(0)
        mask=(positions<lengths.unsqueeze(1)).unsqueeze(-1).float()
        mean=(x.float()*mask).sum(1)/mask.sum(1).clamp_min(1.0)
        features=torch.cat([last,mean],dim=-1)
        return F.normalize(self.projection(features).float(),dim=-1)

def load_archa_model():
    try: payload=torch.load(BASE_CHECKPOINT,map_location='cpu',weights_only=False,mmap=True)
    except Exception: payload=torch.load(BASE_CHECKPOINT,map_location='cpu',weights_only=False)
    cfg=dict(extract_cfg(payload) or MODEL_CONFIG)
    state=extract_state(payload)
    if state is None: raise RuntimeError('Selected Architecture-A checkpoint has no usable state dictionary.')
    state=dict(state)
    if state and all(str(k).startswith('module.') for k in state): state={str(k)[7:]:v for k,v in state.items()}
    backbone=LegalDecoderA(cfg)
    missing,unexpected=backbone.load_state_dict(state,strict=False)
    allowed_missing=[k for k in missing if k.endswith('lm_head.weight') and cfg.get('tie_embeddings',False)]
    bad_missing=[k for k in missing if k not in allowed_missing]
    if bad_missing or unexpected:
        raise RuntimeError(f'Architecture-A checkpoint mismatch: missing={bad_missing[:8]}, unexpected={unexpected[:8]}')
    if int(cfg['d_model'])!=1024 or int(cfg['n_layers'])!=18:
        raise RuntimeError(f'Unexpected Architecture-A dimensions: {cfg.get("d_model")} x {cfg.get("n_layers")}')
    for p in backbone.parameters(): p.requires_grad=True
    backbone.cfg['gradient_checkpointing']=True
    model=RetrievalA(backbone,dim=int(cfg['d_model']))
    del payload,state
    gc.collect()
    return model,cfg

# Test the exact intended T4 microbatch (four sequences x 256 tokens), forward, backward and AdamW step.
_smoke_model,_smoke_cfg=load_archa_model()
_smoke_params=sum(p.numel() for p in _smoke_model.backbone.parameters())
assert _smoke_params==243_897_344, f'Unexpected Phase-1 A parameter count: {_smoke_params:,}'
_smoke_model.to(DEVICE).train()
_smoke_ids=torch.randint(0,40000,(1+2+1, DOC_CHUNK_TOKENS),device=DEVICE)
_smoke_lens=torch.tensor([256,247,231,256],device=DEVICE)
_smoke_opt=torch.optim.AdamW(_smoke_model.parameters(),lr=1e-5,weight_decay=0.01)
_smoke_opt.zero_grad(set_to_none=True)
with torch.autocast('cuda',dtype=torch.float16,enabled=True):
    _smoke_vec=_smoke_model(_smoke_ids,_smoke_lens)
    _smoke_loss=(1.0-(_smoke_vec[0]*_smoke_vec[1]).sum()) + F.softplus(0.15-(_smoke_vec[0]*_smoke_vec[1]).sum()+(_smoke_vec[0]*_smoke_vec[2]).sum())
assert torch.isfinite(_smoke_loss) and torch.isfinite(_smoke_vec).all()
_smoke_loss.backward()
_smoke_grad=torch.nn.utils.clip_grad_norm_([p for p in _smoke_model.parameters() if p.requires_grad],1.0)
assert torch.isfinite(_smoke_grad)
_smoke_opt.step()
print('PASS | A full-finetune forward/backward/AdamW smoke | seq=4 x 256 | backbone params=',f'{_smoke_params:,}','| embedding=',tuple(_smoke_vec.shape),'| loss=',float(_smoke_loss.detach().cpu()))
del _smoke_model,_smoke_opt,_smoke_ids,_smoke_lens,_smoke_vec,_smoke_loss,_smoke_grad
import gc; gc.collect(); torch.cuda.empty_cache()


In [ ]:

def read_json(path):
    with open(path,'r',encoding='utf-8') as f: return json.load(f)

def rows_from(raw):
    if isinstance(raw,list): return raw
    if isinstance(raw,dict):
        for key in ('data','records','cases','queries','candidates','items'):
            if isinstance(raw.get(key),list): return raw[key]
        rows=[]
        for key,val in raw.items():
            if isinstance(val,dict): rows.append(dict(val,id=val.get('id',key)))
            elif isinstance(val,str): rows.append({'id':key,'text':val})
        if rows: return rows
    raise ValueError('Unknown official JSON shape: '+str(type(raw)))

def row_id(row):
    for key in ('id','case_id','doc_id','caseId','document_id','caseID'):
        if row.get(key) is not None: return str(row[key])
    return None

def sentence_text(x):
    if isinstance(x,str): return x
    if isinstance(x,dict):
        for key in ('text','sentence','content','value'):
            if isinstance(x.get(key),str): return x[key]
    return ''

def row_text(row):
    for key in ('text','document','case_text','content','sentences'):
        val=row.get(key)
        if isinstance(val,str) and val.strip(): return re.sub(r'\s+',' ',val).strip()
        if isinstance(val,list):
            joined=' '.join(t for t in (sentence_text(x) for x in val) if t)
            if joined.strip(): return re.sub(r'\s+',' ',joined).strip()
        if isinstance(val,dict):
            for sub in ('sentences','text','content'):
                subval=val.get(sub)
                if isinstance(subval,str) and subval.strip(): return re.sub(r'\s+',' ',subval).strip()
                if isinstance(subval,list):
                    joined=' '.join(t for t in (sentence_text(x) for x in subval) if t)
                    if joined.strip(): return re.sub(r'\s+',' ',joined).strip()
    return ''

def relevant_ids(row):
    for key in ('relevant_candidates','relevant_candidate_ids','relevant_cases','prior_cases'):
        val=row.get(key)
        if isinstance(val,list):
            out=[]
            for x in val:
                if isinstance(x,dict): x=next((x.get(k) for k in ('id','case_id','candidate_id','doc_id') if x.get(k) is not None),None)
                if x is not None: out.append(str(x))
            return list(dict.fromkeys(out))
    return []

def load_split(split):
    paths={'train':('Train/train.json','Train/train_candidates.json'),
           'dev':('Dev/dev.json','Dev/dev_candidates.json'),
           'test':('Test/test.json','Test/test_candidates.json')}
    qfile,cfile=paths[split]
    qrows=rows_from(read_json(DATA_ROOT/qfile)); crows=rows_from(read_json(DATA_ROOT/cfile))
    queries=[]; qseen=set()
    for row in qrows:
        qid=row_id(row); txt=row_text(row)
        if qid and txt:
            if qid in qseen: raise ValueError(f'{split}: duplicate query ID {qid}')
            queries.append({'id':qid,'text':txt,'relevant':relevant_ids(row)}); qseen.add(qid)
    candidates={}
    for row in crows:
        cid=row_id(row); txt=row_text(row)
        if cid and txt:
            if cid in candidates: continue
            candidates[cid]={'id':cid,'text':txt}
    if not queries or len(candidates)<7: raise ValueError(f'{split}: parsed {len(queries)} queries / {len(candidates)} candidates; schema mismatch.')
    if split!='test' and not any(q['relevant'] for q in queries):
        raise ValueError(f'{split}: no relevance labels found.')
    return queries,candidates

TRAIN_Q,TRAIN_C=load_split('train')
DEV_Q,DEV_C=load_split('dev')
TEST_Q,TEST_C=load_split('test')
train_pool_ids=set(TRAIN_C); dev_pool_ids=set(DEV_C); test_pool_ids=set(TEST_C)
train_positive=sum(bool(set(q['relevant'])&train_pool_ids) for q in TRAIN_Q)
dev_labeled=sum(bool(q['relevant']) for q in DEV_Q)
print('TRAIN queries/candidates:',len(TRAIN_Q),len(TRAIN_C),'| positive-overlap queries:',train_positive)
print('DEV   queries/candidates:',len(DEV_Q),len(DEV_C),'| labeled queries:',dev_labeled)
print('TEST  queries/candidates:',len(TEST_Q),len(TEST_C),'| test labels expected absent:',sum(not q['relevant'] for q in TEST_Q))
print('Sample train ID:',TRAIN_Q[0]['id'],'| sample relevant IDs:',TRAIN_Q[0]['relevant'][:5])
assert train_positive>0 and dev_labeled>0
assert len(TEST_C)>=7
assert all(len(C)-int(q['id'] in C)>=7 for q in TEST_Q for C in [test_pool_ids]), 'A test query has fewer than seven non-self candidates.'

# Hash all IDs, texts and train/dev labels so a resumed run cannot silently change datasets.
def split_fingerprint(queries,candidates):
    h=hashlib.sha256()
    for q in queries:
        h.update(q['id'].encode()); h.update(q['text'].encode('utf-8','ignore'))
        h.update('\0'.join(q['relevant']).encode())
    for cid,row in candidates.items():
        h.update(cid.encode()); h.update(row['text'].encode('utf-8','ignore'))
    return h.hexdigest()
DATASET_FINGERPRINT=hashlib.sha256(''.join([
    split_fingerprint(TRAIN_Q,TRAIN_C),split_fingerprint(DEV_Q,DEV_C),split_fingerprint(TEST_Q,TEST_C)
]).encode()).hexdigest()
DATA_INVENTORY={'train_queries':len(TRAIN_Q),'train_candidates':len(TRAIN_C),'train_positive_overlap':train_positive,
                'dev_queries':len(DEV_Q),'dev_candidates':len(DEV_C),'dev_labeled_queries':dev_labeled,
                'test_queries':len(TEST_Q),'test_candidates':len(TEST_C),'dataset_fingerprint':DATASET_FINGERPRINT}
atomic_json(DATA_INVENTORY,ROOT/'data_inventory.json')
# These counts match the successful Phase-7 preflight output supplied by the user; warn, do not hard-fail, if the hosted dataset changes.
_expected=(827,4320,118,1023,237,1727)
_observed=(len(TRAIN_Q),len(TRAIN_C),len(DEV_Q),len(DEV_C),len(TEST_Q),len(TEST_C))
print('Dataset counts versus previously observed counts:',_observed,'| expected baseline:',_expected,'| match:',_observed==_expected)
print('Dataset fingerprint:',DATASET_FINGERPRINT)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import joblib

# Legal cases are represented by three bounded windows: opening, middle, and ending.
def text_spans(text,n_chunks=DOC_CHUNKS,chars_per_span=6500):
    text=str(text or '')
    if len(text)<=chars_per_span: return [text]+['']*(n_chunks-1)
    starts=[0,max(0,(len(text)-chars_per_span)//2),max(0,len(text)-chars_per_span)]
    spans=[]
    for st in starts[:n_chunks]:
        sp=text[st:st+chars_per_span]
        if sp and sp not in spans: spans.append(sp)
    while len(spans)<n_chunks: spans.append('')
    return spans[:n_chunks]

def encode_doc_chunks(text):
    arr=np.zeros((DOC_CHUNKS,DOC_CHUNK_TOKENS),dtype=np.uint16)
    lens=np.zeros((DOC_CHUNKS,),dtype=np.uint16)
    for j,span in enumerate(text_spans(text)):
        if not span: continue
        ids=TOKENIZER.encode(span).ids[:DOC_CHUNK_TOKENS]
        if ids:
            lens[j]=len(ids); arr[j,:len(ids)]=np.asarray(ids,dtype=np.uint16)
    if lens.max()==0:
        ids=TOKENIZER.encode(str(text)[:3000]).ids[:DOC_CHUNK_TOKENS]
        if ids: lens[0]=len(ids); arr[0,:len(ids)]=np.asarray(ids,dtype=np.uint16)
    return arr,lens

def cache_fingerprint(rows):
    h=hashlib.sha256(); h.update(TOKENIZER_SHA256.encode()); h.update(str((len(rows),DOC_CHUNKS,DOC_CHUNK_TOKENS)).encode())
    for row in rows:
        h.update(row['id'].encode()); h.update(row['text'].encode('utf-8','ignore'))
    return h.hexdigest()

class DocTokenCache:
    def __init__(self,name,rows):
        self.name=name; self.rows=rows; self.dir=CACHE_ROOT/name
        self.dir.mkdir(parents=True,exist_ok=True)
        self.tokens_path=self.dir/'tokens.npy'; self.lengths_path=self.dir/'lengths.npy'
        self.complete_state=self.dir/'tokenize_state.json'
        self.build_state=self.dir/'tokenize_build_state.json'
        self.build_tokens=self.dir/'tokens.build.npy'; self.build_lengths=self.dir/'lengths.build.npy'
        self.fingerprint=cache_fingerprint(rows); n=len(rows)
        valid=False
        if self.tokens_path.is_file() and self.lengths_path.is_file() and self.complete_state.is_file():
            try:
                st=json.loads(self.complete_state.read_text())
                tok=np.load(self.tokens_path,mmap_mode='r'); lens=np.load(self.lengths_path,mmap_mode='r')
                valid=(st.get('fingerprint')==self.fingerprint and st.get('done')==n and tok.shape==(n,DOC_CHUNKS,DOC_CHUNK_TOKENS) and lens.shape==(n,DOC_CHUNKS))
                del tok,lens
            except Exception: valid=False
        if not valid:
            # Resume a partial cache only if its own state fingerprint and both array shapes match.
            build_ok=False; done=0
            if self.build_tokens.is_file() and self.build_lengths.is_file() and self.build_state.is_file():
                try:
                    st=json.loads(self.build_state.read_text())
                    tok=np.load(self.build_tokens,mmap_mode='r'); lens=np.load(self.build_lengths,mmap_mode='r')
                    build_ok=(st.get('fingerprint')==self.fingerprint and st.get('total')==n and
                              tok.shape==(n,DOC_CHUNKS,DOC_CHUNK_TOKENS) and lens.shape==(n,DOC_CHUNKS))
                    if build_ok: done=int(st.get('done',0))
                    del tok,lens
                except Exception: build_ok=False
            if not build_ok:
                for p in (self.build_tokens,self.build_lengths,self.build_state): p.unlink(missing_ok=True)
                tok=np.lib.format.open_memmap(self.build_tokens,mode='w+',dtype=np.uint16,shape=(n,DOC_CHUNKS,DOC_CHUNK_TOKENS))
                lens=np.lib.format.open_memmap(self.build_lengths,mode='w+',dtype=np.uint16,shape=(n,DOC_CHUNKS))
                tok[:]=0; lens[:]=0; tok.flush(); lens.flush(); del tok,lens
                done=0
            tok=np.load(self.build_tokens,mmap_mode='r+'); lens=np.load(self.build_lengths,mmap_mode='r+')
            print(f'Tokenizing {name}: {done:,}/{n:,}')
            for start in range(done,n,128):
                stop=min(n,start+128)
                for i in range(start,stop): tok[i],lens[i]=encode_doc_chunks(rows[i]['text'])
                tok.flush(); lens.flush()
                atomic_json({'fingerprint':self.fingerprint,'done':stop,'total':n},self.build_state)
                if stop%1024==0 or stop==n: print(f'  {name}: {stop:,}/{n:,}',flush=True)
            del tok,lens
            os.replace(self.build_tokens,self.tokens_path); os.replace(self.build_lengths,self.lengths_path)
            atomic_json({'fingerprint':self.fingerprint,'done':n,'total':n},self.complete_state)
            self.build_state.unlink(missing_ok=True)
        self.tokens=np.load(self.tokens_path,mmap_mode='r')
        self.lengths=np.load(self.lengths_path,mmap_mode='r')
        self.id_to_idx={row['id']:i for i,row in enumerate(rows)}
        print('CACHE READY:',name,self.tokens.shape)

TOKEN_CACHES={
 'train_q':DocTokenCache('train_queries',TRAIN_Q),
 'train_c':DocTokenCache('train_candidates',list(TRAIN_C.values())),
 'dev_q':DocTokenCache('dev_queries',DEV_Q),
 'dev_c':DocTokenCache('dev_candidates',list(DEV_C.values())),
 'test_q':DocTokenCache('test_queries',TEST_Q),
 'test_c':DocTokenCache('test_candidates',list(TEST_C.values())),
}

# Lexical TF-IDF is fitted only on the official training candidate pool.
LEX_DIR=CACHE_ROOT/'lexical_index'; LEX_DIR.mkdir(parents=True,exist_ok=True)
VECTORIZER_PATH=LEX_DIR/'tfidf_vectorizer.joblib'
TRAIN_X_PATH=LEX_DIR/'train_candidates_tfidf.npz'
LEX_META_PATH=LEX_DIR/'lexical_manifest.json'
LEX_FP=hashlib.sha256((DATASET_FINGERPRINT+TOKENIZER_SHA256+'tfidf-v1').encode()).hexdigest()
lex_ok=False
if VECTORIZER_PATH.is_file() and TRAIN_X_PATH.is_file() and LEX_META_PATH.is_file():
    try:
        meta=json.loads(LEX_META_PATH.read_text()); lex_ok=meta.get('fingerprint')==LEX_FP
    except Exception: lex_ok=False
if lex_ok:
    vectorizer=joblib.load(VECTORIZER_PATH); train_X=sparse.load_npz(TRAIN_X_PATH).tocsr()
    if train_X.shape[0]!=len(TRAIN_C): lex_ok=False
if not lex_ok:
    vectorizer=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=0.98,max_features=150_000,
        sublinear_tf=True,norm='l2',dtype=np.float32,token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z0-9]+\b')
    train_X=vectorizer.fit_transform([x['text'] for x in TRAIN_C.values()]).tocsr()
    tmp_x=TRAIN_X_PATH.with_suffix('.tmp.npz'); sparse.save_npz(tmp_x,train_X,compressed=True); os.replace(tmp_x,TRAIN_X_PATH)
    tmp_v=VECTORIZER_PATH.with_suffix('.tmp'); joblib.dump(vectorizer,tmp_v,compress=3); os.replace(tmp_v,VECTORIZER_PATH)
    atomic_json({'fingerprint':LEX_FP,'shape':train_X.shape,'features':len(vectorizer.vocabulary_)},LEX_META_PATH)
    print('Built train-pool TF-IDF:',train_X.shape)
else:
    print('Reused train-pool TF-IDF:',train_X.shape)

# Precompute Dev lexical matrices once; tie them to the exact dataset fingerprint.
DEV_CX_PATH=LEX_DIR/'dev_candidates_tfidf.npz'; DEV_QX_PATH=LEX_DIR/'dev_queries_tfidf.npz'
DEV_LEX_META=LEX_DIR/'dev_lexical_manifest.json'
dev_lex_ok=False
if DEV_CX_PATH.is_file() and DEV_QX_PATH.is_file() and DEV_LEX_META.is_file():
    try:
        dm=json.loads(DEV_LEX_META.read_text())
        if dm.get('dataset_fingerprint')==DATASET_FINGERPRINT and dm.get('lexical_fingerprint')==LEX_FP:
            DEV_CX=sparse.load_npz(DEV_CX_PATH).tocsr(); DEV_QX=sparse.load_npz(DEV_QX_PATH).tocsr()
            dev_lex_ok=(DEV_CX.shape[0]==len(DEV_C) and DEV_QX.shape[0]==len(DEV_Q))
    except Exception: dev_lex_ok=False
if not dev_lex_ok:
    DEV_CX=vectorizer.transform([x['text'] for x in DEV_C.values()]).tocsr()
    DEV_QX=vectorizer.transform([x['text'] for x in DEV_Q]).tocsr()
    tmp_dc=DEV_CX_PATH.with_suffix('.tmp.npz'); tmp_dq=DEV_QX_PATH.with_suffix('.tmp.npz')
    sparse.save_npz(tmp_dc,DEV_CX,compressed=True); sparse.save_npz(tmp_dq,DEV_QX,compressed=True)
    os.replace(tmp_dc,DEV_CX_PATH); os.replace(tmp_dq,DEV_QX_PATH)
    atomic_json({'dataset_fingerprint':DATASET_FINGERPRINT,'lexical_fingerprint':LEX_FP,
                 'candidate_rows':DEV_CX.shape[0],'query_rows':DEV_QX.shape[0]},DEV_LEX_META)

HARDNEG_PATH=LEX_DIR/f'train_hard_negatives_{DATASET_FINGERPRINT[:12]}.jsonl'
hard_negatives={}
if HARDNEG_PATH.is_file():
    with open(HARDNEG_PATH,encoding='utf-8') as f:
        for line in f:
            try:
                obj=json.loads(line); hard_negatives[obj['query_id']]=obj['negative_ids']
            except Exception: pass
train_c_ids=list(TRAIN_C.keys())
needs=[q for q in TRAIN_Q if q['id'] not in hard_negatives and set(q['relevant'])&set(TRAIN_C)]
print('Hard-negative cache:',len(hard_negatives),'queries | remaining:',len(needs))
with open(HARDNEG_PATH,'a',encoding='utf-8') as f:
    for start in range(0,len(needs),16):
        batch=needs[start:start+16]
        qmat=vectorizer.transform([q['text'] for q in batch])
        sims=(qmat@train_X.T).toarray()
        for q,score in zip(batch,sims):
            k=min(120,len(score)); top=np.argpartition(score,-k)[-k:]; top=top[np.argsort(score[top])[::-1]]
            positives=set(q['relevant']); neg=[]
            for ix in top:
                cid=train_c_ids[int(ix)]
                if cid not in positives and cid!=q['id']: neg.append(cid)
                if len(neg)>=80: break
            hard_negatives[q['id']]=neg
            f.write(json.dumps({'query_id':q['id'],'negative_ids':neg})+'\n')
        f.flush()
        if start+len(batch)==len(needs) or (start+len(batch))%256==0:
            print('Hard negatives:',len(hard_negatives),'/',len(TRAIN_Q),flush=True)

TRAIN_GROUP_QUERIES=[q for q in TRAIN_Q if any(cid in TRAIN_C for cid in q['relevant'])]
if not TRAIN_GROUP_QUERIES: raise RuntimeError('No train queries have a relevant case in the official train candidate pool.')
atomic_json({'dataset_fingerprint':DATASET_FINGERPRINT,'train_queries':len(TRAIN_Q),'usable_positive_queries':len(TRAIN_GROUP_QUERIES),
    'hard_negative_queries':len(hard_negatives),'doc_chunks':DOC_CHUNKS,'chunk_tokens':DOC_CHUNK_TOKENS,
    'tfidf_features':len(vectorizer.vocabulary_)},ROOT/'preprocessing_manifest.json')
print('Preprocessing complete | usable training queries:',len(TRAIN_GROUP_QUERIES))


In [ ]:
EXP_NAME='A_FullFineTune_LongRun'
EXP_ROOT=EXP_DIR
EXP_ROOT.mkdir(parents=True,exist_ok=True); SUB_DIR.mkdir(parents=True,exist_ok=True)
LATEST_RESUME=EXP_ROOT/'latest_resume.pt'
BEST_MODEL=EXP_ROOT/'best_model.pt'
BEST_METRICS=EXP_ROOT/'best_dev_metrics.json'
COMPLETE_MARKER=EXP_ROOT/'experiment_complete.json'
TRAIN_BUDGET_SEC=TOTAL_TRAIN_HOURS*3600


def tree_to_cpu(obj):
    if torch.is_tensor(obj): return obj.detach().cpu()
    if isinstance(obj,dict): return {k:tree_to_cpu(v) for k,v in obj.items()}
    if isinstance(obj,list): return [tree_to_cpu(v) for v in obj]
    if isinstance(obj,tuple): return tuple(tree_to_cpu(v) for v in obj)
    return obj

def cpu_model_state(model):
    # Preserve FP32 master weights. Casting to FP16 here would introduce irreversible drift on every resume.
    return {k:v.detach().cpu().contiguous() for k,v in model.state_dict().items()}

def save_resume_checkpoint(model,optimizer,scheduler,scaler,step,train_elapsed,best_dev_score,best_alpha,last_eval_elapsed,extra=None):
    free=free_gb()
    if free<MIN_FREE_GB_FOR_RESUME_SAVE:
        print(f'CHECKPOINT NOT WRITTEN: only {free:.2f} GiB free; previous atomic checkpoint is preserved.')
        return False
    EXP_ROOT.mkdir(parents=True,exist_ok=True)
    payload={
        'magic':RUN_MAGIC,'experiment':EXP_NAME,'architecture':'A','method':'full_fine_tuning',
        'tokenizer_sha256':TOKENIZER_SHA256,'base_checkpoint':str(BASE_CHECKPOINT),'base_checkpoint_sha256':BASE_CHECKPOINT_SHA256,
        'dataset_fingerprint':DATASET_FINGERPRINT,'model_config':MODEL_CONFIG,
        'model_state_dict':cpu_model_state(model),
        'optimizer_state_dict':tree_to_cpu(optimizer.state_dict()),
        'scheduler_state_dict':scheduler.state_dict(),
        'scaler_state_dict':scaler.state_dict() if scaler is not None else None,
        'step':int(step),'train_elapsed_seconds':float(train_elapsed),'best_dev_micro_f1_at7':float(best_dev_score),
        'best_alpha':float(best_alpha),'last_eval_train_seconds':float(last_eval_elapsed),
        'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),
        'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'timestamp':time.strftime('%Y-%m-%d %H:%M:%S'),'extra':extra or {},
    }
    atomic_torch_save(payload,LATEST_RESUME)
    atomic_json({'magic':RUN_MAGIC,'experiment':EXP_NAME,'step':int(step),'train_elapsed_seconds':float(train_elapsed),
                 'latest_resume':str(LATEST_RESUME),'base_checkpoint_sha256':BASE_CHECKPOINT_SHA256,
                 'dataset_fingerprint':DATASET_FINGERPRINT,'timestamp':payload['timestamp']},EXP_ROOT/'resume_manifest.json')
    del payload
    print('ATOMIC RESUME CHECKPOINT SAVED | step=',step,'| size GiB=',round(LATEST_RESUME.stat().st_size/(1024**3),2),'| free GiB=',round(free_gb(),2))
    return True

def load_resume_checkpoint(model,optimizer,scheduler,scaler):
    if not LATEST_RESUME.is_file(): return 0,0.0,-1.0,0.20,0.0
    print('Loading resume checkpoint:',LATEST_RESUME,'| GiB:',round(LATEST_RESUME.stat().st_size/(1024**3),2))
    p=torch.load(LATEST_RESUME,map_location='cpu',weights_only=False)
    checks={
      'magic':p.get('magic')==RUN_MAGIC,
      'experiment':p.get('experiment')==EXP_NAME,
      'tokenizer':p.get('tokenizer_sha256')==TOKENIZER_SHA256,
      'base_checkpoint':p.get('base_checkpoint_sha256')==BASE_CHECKPOINT_SHA256,
      'dataset':p.get('dataset_fingerprint')==DATASET_FINGERPRINT,
      'model_config':p.get('model_config')==MODEL_CONFIG,
    }
    failed=[k for k,v in checks.items() if not v]
    if failed: raise RuntimeError('Resume checkpoint incompatible; failed checks: '+', '.join(failed))
    model.load_state_dict(p['model_state_dict'],strict=True)
    optimizer.load_state_dict(p['optimizer_state_dict'])
    scheduler.load_state_dict(p['scheduler_state_dict'])
    if scaler is not None and p.get('scaler_state_dict'): scaler.load_state_dict(p['scaler_state_dict'])
    random.setstate(p['python_rng']); np.random.set_state(p['numpy_rng']); torch.set_rng_state(p['torch_rng'])
    saved_cuda=p.get('cuda_rng')
    if torch.cuda.is_available() and saved_cuda:
        for i,state in enumerate(saved_cuda[:torch.cuda.device_count()]): torch.cuda.set_rng_state(state,device=i)
    out=(int(p['step']),float(p['train_elapsed_seconds']),float(p.get('best_dev_micro_f1_at7',-1.0)),
         float(p.get('best_alpha',0.20)),float(p.get('last_eval_train_seconds',0.0)))
    del p; gc.collect()
    return out

train_c_ids=list(TRAIN_C.keys())

def draw_training_group(query):
    positives=[cid for cid in query['relevant'] if cid in TRAIN_C and cid!=query['id']]
    if not positives: return None
    positive=random.choice(positives)
    hard=[cid for cid in hard_negatives.get(query['id'],[]) if cid in TRAIN_C and cid not in set(positives) and cid!=query['id']]
    chosen=[]
    if hard: chosen.append(random.choice(hard))
    while len(chosen)<NEGATIVES_PER_QUERY:
        cid=random.choice(train_c_ids)
        if cid!=query['id'] and cid not in positives and cid not in chosen: chosen.append(cid)
    return query['id'],positive,chosen

def sample_chunk(cache,doc_id):
    i=cache.id_to_idx[doc_id]
    valid=np.flatnonzero(cache.lengths[i]>0)
    if len(valid)==0: raise RuntimeError(f'No tokenized document chunks for {doc_id}')
    j=int(random.choice(valid.tolist()))
    return np.asarray(cache.tokens[i,j],dtype=np.int64),int(cache.lengths[i,j])

def make_group_batch(batch_size=GROUP_BATCH_SIZE):
    chosen=[]
    while len(chosen)<batch_size:
        q=random.choice(TRAIN_GROUP_QUERIES)
        group=draw_training_group(q)
        if group is not None: chosen.append(group)
    ids=[]; lengths=[]
    for qid,pos,negs in chosen:
        for cid,cache in [(qid,TOKEN_CACHES['train_q']),(pos,TOKEN_CACHES['train_c'])]+[(n,TOKEN_CACHES['train_c']) for n in negs]:
            tok,length=sample_chunk(cache,cid); ids.append(tok); lengths.append(length)
    width=max(lengths); x=np.zeros((len(ids),width),dtype=np.int64)
    for i,(tok,length) in enumerate(zip(ids,lengths)): x[i,:length]=tok[:length]
    return torch.tensor(x,device=DEVICE,dtype=torch.long),torch.tensor(lengths,device=DEVICE,dtype=torch.long),chosen

def make_optimizer(model):
    params=[p for p in model.parameters() if p.requires_grad]
    try: optimizer=torch.optim.AdamW(params,lr=TRAIN_LR,betas=(0.9,0.95),weight_decay=WEIGHT_DECAY,eps=1e-8,fused=True)
    except Exception: optimizer=torch.optim.AdamW(params,lr=TRAIN_LR,betas=(0.9,0.95),weight_decay=WEIGHT_DECAY,eps=1e-8)
    return optimizer,params

@torch.inference_mode()
def encode_cache_eval(model,cache,batch_sequences=EVAL_BATCH_SEQUENCES):
    n=len(cache.rows); d=int(MODEL_CONFIG['d_model']); total=n*DOC_CHUNKS
    token_view=cache.tokens.reshape(total,DOC_CHUNK_TOKENS)
    length_view=cache.lengths.reshape(total)
    valid_idx=np.flatnonzero(np.asarray(length_view)>0)
    vectors=np.zeros((total,d),dtype=np.float16)
    model.eval()
    for start in range(0,len(valid_idx),batch_sequences):
        ids=valid_idx[start:start+batch_sequences]
        x_np=np.asarray(token_view[ids],dtype=np.int64)
        l_np=np.asarray(length_view[ids],dtype=np.int64)
        x=torch.as_tensor(x_np.copy(),device=DEVICE,dtype=torch.long)
        lengths=torch.as_tensor(l_np.copy(),device=DEVICE,dtype=torch.long)
        with torch.autocast('cuda',dtype=torch.float16,enabled=True): z=model(x,lengths).float()
        vectors[ids]=z.detach().cpu().numpy().astype(np.float16)
        if start and start%512==0: print('  embedding chunks:',start,'/',len(valid_idx),flush=True)
    views=vectors.reshape(n,DOC_CHUNKS,d).astype(np.float32)
    mask=(np.asarray(cache.lengths)>0).astype(np.float32)
    pooled=(views*mask[:,:,None]).sum(1)/np.maximum(mask.sum(1,keepdims=True),1.0)
    pooled/=np.maximum(np.linalg.norm(pooled,axis=1,keepdims=True),1e-8)
    return pooled.astype(np.float32)

def predictions_from_scores(scores,queries,candidates):
    cand_ids=list(candidates.keys()); positions={cid:i for i,cid in enumerate(cand_ids)}; out={}
    if len(cand_ids)<7: raise ValueError('Candidate pool has fewer than seven cases.')
    scores=np.asarray(scores,dtype=np.float32).copy()
    for i,q in enumerate(queries):
        self_idx=positions.get(q['id'])
        if self_idx is not None: scores[i,self_idx]=-np.inf
        if np.isfinite(scores[i]).sum()<7: raise ValueError(f'{q["id"]}: fewer than seven eligible candidate cases.')
        idx=np.argpartition(scores[i],-7)[-7:]
        idx=idx[np.argsort(scores[i,idx])[::-1]]
        out[q['id']]=[cand_ids[int(j)] for j in idx]
    return out

def evaluate_dev_model(model):
    # Full official Dev candidate pool and all Dev queries; no sampling in model selection.
    c_emb=encode_cache_eval(model,TOKEN_CACHES['dev_c'])
    q_emb=encode_cache_eval(model,TOKEN_CACHES['dev_q'])
    neural=q_emb@c_emb.T
    lexical=(DEV_QX@DEV_CX.T).toarray().astype(np.float32)
    neural=(neural-neural.mean(1,keepdims=True))/(neural.std(1,keepdims=True)+1e-6)
    lexical=(lexical-lexical.mean(1,keepdims=True))/(lexical.std(1,keepdims=True)+1e-6)
    results=[]
    for alpha in (0.0,0.15,0.30,0.45):
        pred=predictions_from_scores((1-alpha)*neural+alpha*lexical,DEV_Q,DEV_C)
        metrics=micro_metrics(pred,DEV_Q)
        results.append({'alpha':alpha,**metrics})
    best=max(results,key=lambda x:x['micro_f1@7'])
    return best,results

def save_best_model(model,step,metrics):
    if free_gb()<2.0:
        print('BEST MODEL NOT WRITTEN: less than 2 GiB free; retaining previous best checkpoint.')
        return False
    payload={'magic':RUN_MAGIC,'architecture':'A','method':'full_fine_tuning','step':int(step),
        'tokenizer_sha256':TOKENIZER_SHA256,'base_checkpoint_sha256':BASE_CHECKPOINT_SHA256,
        'dataset_fingerprint':DATASET_FINGERPRINT,'model_config':MODEL_CONFIG,
        'model_state_dict':cpu_model_state(model),'dev_metrics':metrics}
    atomic_torch_save(payload,BEST_MODEL)
    atomic_json({'step':int(step),'dev_metrics':metrics,'best_model':str(BEST_MODEL)},BEST_METRICS)
    del payload
    print('BEST DEV MODEL SAVED | step=',step,'| Micro-F1@7=',round(metrics['micro_f1@7'],6))
    return True

# If the completed run is opened again, restore its selected model for the final audit; otherwise resume training.
retrieval_model=None; optimizer=None; scheduler=None; scaler=None
if COMPLETE_MARKER.is_file() and BEST_MODEL.is_file():
    print('Completed Architecture-A experiment found; loading its selected best model.')
    retrieval_model,_cfg=load_archa_model(); retrieval_model.to(DEVICE)
    best_payload=torch.load(BEST_MODEL,map_location='cpu',weights_only=False)
    if best_payload.get('magic')!=RUN_MAGIC or best_payload.get('base_checkpoint_sha256')!=BASE_CHECKPOINT_SHA256 or best_payload.get('dataset_fingerprint')!=DATASET_FINGERPRINT:
        raise RuntimeError('Completed best-model artifact does not match the current checkpoint/dataset.')
    retrieval_model.load_state_dict(best_payload['model_state_dict'],strict=True)
    del best_payload; gc.collect()
    training_state=json.loads((EXP_ROOT/'training_state.json').read_text())
    start_step=int(training_state.get('steps',0)); train_elapsed=float(training_state.get('training_hours',0))*3600
    best_dev_score=float(training_state.get('best_dev_micro_f1_at7',-1.0)); best_alpha=float(training_state.get('best_alpha',0.20))
else:
    retrieval_model,_cfg=load_archa_model(); retrieval_model.to(DEVICE)
    optimizer,trainable_params=make_optimizer(retrieval_model)
    total_schedule_steps=40_000
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lambda step: min(1.0,(step+1)/500.0) if step<500 else (0.10+0.90*0.5*(1.0+math.cos(math.pi*min(1.0,(step-500)/(total_schedule_steps-500))))))
    try: scaler=torch.amp.GradScaler('cuda',enabled=True)
    except Exception: scaler=torch.cuda.amp.GradScaler(enabled=True)
    start_step,train_elapsed,best_dev_score,best_alpha,last_eval_elapsed=load_resume_checkpoint(retrieval_model,optimizer,scheduler,scaler)
    print('Trainable params:',sum(p.numel() for p in trainable_params),'| resume step:',start_step,'| training hours:',round(train_elapsed/3600,2),'| best Dev F1:',best_dev_score)
    # If a best-model artifact exists but latest checkpoint predates its metric, recover its best score.
    if BEST_METRICS.is_file():
        try:
            m=json.loads(BEST_METRICS.read_text())
            stored_score=float(m['dev_metrics']['micro_f1@7'])
            if stored_score>=best_dev_score:
                best_dev_score=stored_score; best_alpha=float(m['dev_metrics']['alpha'])
        except Exception: pass
    stop_reason='budget_or_session_limit'; last_saved_step=start_step; last_saved_elapsed=train_elapsed
    recent_losses=[]; optimizer.zero_grad(set_to_none=True); retrieval_model.train()
    caught_error=None
    try:
        while train_elapsed<TRAIN_BUDGET_SEC and time.time()<SESSION_DEADLINE-SESSION_STOP_RESERVE_MIN*60 and start_step<MAX_TRAIN_STEPS:
            step_start=time.time()
            input_ids,lengths,groups=make_group_batch()
            optimizer.zero_grad(set_to_none=True)
            try:
                with torch.autocast('cuda',dtype=torch.float16,enabled=True):
                    vectors=retrieval_model(input_ids,lengths)
                    stride=2+NEGATIVES_PER_QUERY
                    qv=vectors[0::stride]; pv=vectors[1::stride]
                    nv=torch.stack([vectors[2+i::stride] for i in range(NEGATIVES_PER_QUERY)],dim=1)
                    pos=(qv*pv).sum(-1,keepdim=True)
                    neg=(qv.unsqueeze(1)*nv).sum(-1)
                    scores=torch.cat([pos,neg],dim=1).float()
                    labels=torch.zeros(len(groups),dtype=torch.long,device=DEVICE)
                    contrastive=F.cross_entropy(scores/TEMPERATURE,labels)
                    pairwise=F.softplus(PAIRWISE_MARGIN-pos.float()+neg).mean()
                    loss=contrastive+0.5*pairwise
                if not torch.isfinite(loss): raise FloatingPointError('Non-finite retrieval loss.')
                scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                grad_norm=torch.nn.utils.clip_grad_norm_(trainable_params,1.0)
                old_scale=scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                step_wall=time.time()-step_start; train_elapsed+=step_wall
                if scaler.get_scale()<old_scale:
                    print('GradScaler skipped an optimizer update due to overflow at attempted step',start_step+1,'; retrying with reduced scale.',flush=True)
                    continue
                scheduler.step(); start_step+=1
                recent_losses.append(float(loss.detach().cpu()))
                if len(recent_losses)>100: recent_losses=recent_losses[-100:]
                if start_step%LOG_EVERY_STEPS==0:
                    print(f'{EXP_NAME} | step={start_step:,} | optimizer_h={train_elapsed/3600:.2f}/{TOTAL_TRAIN_HOURS:.2f} | loss={np.mean(recent_losses[-LOG_EVERY_STEPS:]):.5f} | grad={float(grad_norm):.3f} | lr={optimizer.param_groups[0]["lr"]:.2e}',flush=True)
                if start_step%CHECKPOINT_EVERY_STEPS==0:
                    ok=save_resume_checkpoint(retrieval_model,optimizer,scheduler,scaler,start_step,train_elapsed,best_dev_score,best_alpha,last_eval_elapsed,extra={'recent_loss_mean':float(np.mean(recent_losses))})
                    if ok: last_saved_step=start_step; last_saved_elapsed=train_elapsed
                    else:
                        stop_reason='disk_guard'; print('Stopping training to preserve the last complete resume checkpoint.'); break
                # Evaluate on the full Dev split every 2.5 optimizer-hours, with time reserved for the safe stop.
                if train_elapsed-last_eval_elapsed>=EVAL_EVERY_TRAIN_HOURS*3600 and time.time()<SESSION_DEADLINE-(SESSION_STOP_RESERVE_MIN+20)*60:
                    print('Running full Dev evaluation at optimizer-hours:',round(train_elapsed/3600,2),flush=True)
                    retrieval_model.eval(); dev_best,dev_rows=evaluate_dev_model(retrieval_model); retrieval_model.train()
                    atomic_json({'step':start_step,'best':dev_best,'all_alphas':dev_rows},EXP_ROOT/'dev_history_step_'+str(start_step)+'.json')
                    if dev_best['micro_f1@7']>best_dev_score+1e-9:
                        if save_best_model(retrieval_model,start_step,dev_best):
                            best_dev_score=float(dev_best['micro_f1@7']); best_alpha=float(dev_best['alpha'])
                    last_eval_elapsed=train_elapsed
                    ok=save_resume_checkpoint(retrieval_model,optimizer,scheduler,scaler,start_step,train_elapsed,best_dev_score,best_alpha,last_eval_elapsed,extra={'last_dev_eval':dev_best})
                    if ok: last_saved_step=start_step; last_saved_elapsed=train_elapsed
                    else: stop_reason='disk_guard_after_eval'; break
            except torch.cuda.OutOfMemoryError as exc:
                torch.cuda.empty_cache()
                raise RuntimeError(f'CUDA OOM at optimizer step {start_step}. The checkpoint before this step is preserved; lower EMBED_BATCH_SEQUENCES or GROUP_BATCH_SIZE before retrying.') from exc
        # Evaluate the final training state as well; the last interval may precede the 13.5-hour target.
        if train_elapsed>=TRAIN_BUDGET_SEC and train_elapsed-last_eval_elapsed>300:
            print('Final full-Dev evaluation on last training state...',flush=True)
            retrieval_model.eval(); final_dev,final_rows=evaluate_dev_model(retrieval_model); retrieval_model.train()
            atomic_json({'step':start_step,'best':final_dev,'all_alphas':final_rows,'final_training_evaluation':True},EXP_ROOT/'dev_history_final.json')
            if final_dev['micro_f1@7']>best_dev_score+1e-9:
                if save_best_model(retrieval_model,start_step,final_dev):
                    best_dev_score=float(final_dev['micro_f1@7']); best_alpha=float(final_dev['alpha'])
            last_eval_elapsed=train_elapsed
            ok=save_resume_checkpoint(retrieval_model,optimizer,scheduler,scaler,start_step,train_elapsed,best_dev_score,best_alpha,last_eval_elapsed,extra={'final_dev_eval':final_dev})
            if ok: last_saved_step=start_step; last_saved_elapsed=train_elapsed
            else: stop_reason='disk_guard_after_final_eval'
    except KeyboardInterrupt:
        stop_reason='keyboard_interrupt'
        print('Interrupt received; attempting a final atomic checkpoint.')
    except Exception as exc:
        caught_error=exc; stop_reason='error_'+type(exc).__name__
        atomic_json({'error':type(exc).__name__,'message':str(exc),'step':start_step},EXP_ROOT/'last_training_error.json')
        print('TRAINING ERROR:',repr(exc),'| attempting to preserve last state.',flush=True)
    finally:
        # If space prevents a new write, retain the prior checkpoint and report its exact step/time.
        if start_step==last_saved_step:
            saved=save_resume_checkpoint(retrieval_model,optimizer,scheduler,scaler,start_step,train_elapsed,best_dev_score,best_alpha,last_eval_elapsed,extra={'stop_reason':stop_reason})
            if saved: last_saved_step=start_step; last_saved_elapsed=train_elapsed
        else:
            saved=save_resume_checkpoint(retrieval_model,optimizer,scheduler,scaler,start_step,train_elapsed,best_dev_score,best_alpha,last_eval_elapsed,extra={'stop_reason':stop_reason})
            if saved: last_saved_step=start_step; last_saved_elapsed=train_elapsed
        gc.collect(); torch.cuda.empty_cache()
        state={'experiment':EXP_NAME,'architecture':'A','method':'full_fine_tuning','steps':last_saved_step,
            'training_hours':round(last_saved_elapsed/3600,4),'best_dev_micro_f1_at7':best_dev_score,'best_alpha':best_alpha,
            'training_budget_hours':TOTAL_TRAIN_HOURS,'session_stop_reason':stop_reason,
            'latest_resume':str(LATEST_RESUME),'latest_resume_exists':LATEST_RESUME.is_file(),
            'uncheckpointed_steps':max(0,start_step-last_saved_step),'status':'training_complete' if last_saved_elapsed>=TRAIN_BUDGET_SEC else 'paused_resumable'}
        atomic_json(state,EXP_ROOT/'training_state.json')
    if caught_error is not None: raise caught_error
    print('TRAINING SESSION STOPPED SAFELY:',json.dumps(state,indent=2))


In [ ]:
EXP_ROOT.mkdir(parents=True,exist_ok=True); SUB_DIR.mkdir(parents=True,exist_ok=True)

@torch.inference_mode()
def encode_document_cache(model,cache,name,batch_docs=EMBED_BATCH_DOCS,model_step=None):
    out_path=EXP_ROOT/f'{name}_embeddings.npy'; state_path=EXP_ROOT/f'{name}_embedding_state.json'
    n=len(cache.rows); d=int(MODEL_CONFIG['d_model'])
    if model_step is None: model_step=int(json.loads((EXP_ROOT/'training_state.json').read_text()).get('steps',-1))
    model_fingerprint=sha256_file(BEST_MODEL) if BEST_MODEL.is_file() else 'no_best_model_file'
    done=0; valid=False; tmp=out_path.with_suffix('.build.npy')
    if state_path.is_file():
        try:
            st=json.loads(state_path.read_text())
            state_matches=(st.get('model_step')==model_step and st.get('dataset_fingerprint')==DATASET_FINGERPRINT
                           and st.get('model_fingerprint')==model_fingerprint)
            done=int(st.get('done',0)) if state_matches else 0
            if state_matches and out_path.is_file() and done==n:
                old=np.load(out_path,mmap_mode='r')
                valid=(old.shape==(n,d)); del old
                if valid: return np.asarray(np.load(out_path,mmap_mode='r'),dtype=np.float32)
            # Critical restart case: the final .npy does not exist yet, but the .build.npy and progress state do.
            if state_matches and 0<=done<n and tmp.is_file():
                partial=np.load(tmp,mmap_mode='r')
                valid=(partial.shape==(n,d)); del partial
        except Exception: valid=False
    if valid and done<n:
        out=np.load(tmp,mmap_mode='r+')
        print(f'Resuming {name} embeddings at {done:,}/{n:,} documents.')
    else:
        # Missing/incompatible partial array: restart this embedding cache, never use a partial final file.
        tmp.unlink(missing_ok=True); done=0
        out=np.lib.format.open_memmap(tmp,mode='w+',dtype=np.float16,shape=(n,d))
        out[:]=0; out.flush()
    model.eval()
    for start in range(done,n,batch_docs):
        stop=min(n,start+batch_docs)
        token_block=np.asarray(cache.tokens[start:stop],dtype=np.int64)
        length_block=np.asarray(cache.lengths[start:stop],dtype=np.int64)
        valid_mask=length_block>0
        flat_tokens=token_block[valid_mask]; flat_lengths=length_block[valid_mask]
        view_vectors=np.zeros((stop-start,DOC_CHUNKS,d),dtype=np.float32)
        if len(flat_tokens):
            x=torch.as_tensor(flat_tokens.copy(),device=DEVICE,dtype=torch.long)
            lengths=torch.as_tensor(flat_lengths.copy(),device=DEVICE,dtype=torch.long)
            with torch.autocast('cuda',dtype=torch.float16,enabled=True): vec=model(x,lengths).float()
            view_vectors[valid_mask]=vec.detach().cpu().numpy()
        counts=valid_mask.sum(1,keepdims=True).clip(min=1)
        pooled=view_vectors.sum(1)/counts
        pooled/=np.maximum(np.linalg.norm(pooled,axis=1,keepdims=True),1e-8)
        out[start:stop]=pooled.astype(np.float16)
        if stop%128==0 or stop==n:
            out.flush()
            atomic_json({'model_step':model_step,'dataset_fingerprint':DATASET_FINGERPRINT,'model_fingerprint':model_fingerprint,'done':stop,'total':n},state_path)
            print(f'{name} embeddings: {stop:,}/{n:,}',flush=True)
    out.flush(); del out
    os.replace(tmp,out_path)
    atomic_json({'model_step':model_step,'dataset_fingerprint':DATASET_FINGERPRINT,'model_fingerprint':model_fingerprint,'done':n,'total':n},state_path)
    return np.asarray(np.load(out_path,mmap_mode='r'),dtype=np.float32)

def micro_metrics(pred,queries):
    tp=fp=fn=0
    for q in queries:
        gold=set(q['relevant']); got=set(pred.get(q['id'],[])[:7])
        tp+=len(gold&got); fp+=len(got-gold); fn+=len(gold-got)
    precision=tp/(tp+fp) if tp+fp else 0.0; recall=tp/(tp+fn) if tp+fn else 0.0
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
    return {'micro_f1@7':f1,'micro_precision@7':precision,'micro_recall@7':recall,'tp':tp,'fp':fp,'fn':fn}

def tfidf_split(candidates,queries,tag):
    cpath=EXP_ROOT/f'{tag}_c_tfidf.npz'; qpath=EXP_ROOT/f'{tag}_q_tfidf.npz'
    meta_path=EXP_ROOT/f'{tag}_tfidf_manifest.json'
    if cpath.is_file() and qpath.is_file() and meta_path.is_file():
        try:
            meta=json.loads(meta_path.read_text())
            if meta.get('dataset_fingerprint')==DATASET_FINGERPRINT and meta.get('tag')==tag:
                cx=sparse.load_npz(cpath).tocsr(); qx=sparse.load_npz(qpath).tocsr()
                if cx.shape[0]==len(candidates) and qx.shape[0]==len(queries): return cx,qx
        except Exception: pass
    cx=vectorizer.transform([v['text'] for v in candidates.values()]).tocsr()
    qx=vectorizer.transform([v['text'] for v in queries]).tocsr()
    tmp_c=cpath.with_suffix('.tmp.npz'); tmp_q=qpath.with_suffix('.tmp.npz')
    sparse.save_npz(tmp_c,cx,compressed=True); sparse.save_npz(tmp_q,qx,compressed=True)
    os.replace(tmp_c,cpath); os.replace(tmp_q,qpath)
    atomic_json({'dataset_fingerprint':DATASET_FINGERPRINT,'tag':tag,'candidate_rows':cx.shape[0],'query_rows':qx.shape[0]},meta_path)
    return cx,qx

def rank_blend(q_emb,c_emb,qx,cx,queries,candidates,alpha=0.25):
    cand_ids=list(candidates.keys()); cand_pos={cid:i for i,cid in enumerate(cand_ids)}
    if len(cand_ids)<7: raise ValueError('Candidate pool has fewer than seven records.')
    ctorch=torch.as_tensor(c_emb,device=DEVICE,dtype=torch.float32)
    qtorch=torch.as_tensor(q_emb,device=DEVICE,dtype=torch.float32)
    preds={}
    for start in range(0,len(queries),16):
        stop=min(len(queries),start+16)
        neural=(qtorch[start:stop]@ctorch.T).detach().cpu().numpy()
        lexical=(qx[start:stop]@cx.T).toarray().astype(np.float32)
        neural=(neural-neural.mean(1,keepdims=True))/(neural.std(1,keepdims=True)+1e-6)
        lexical=(lexical-lexical.mean(1,keepdims=True))/(lexical.std(1,keepdims=True)+1e-6)
        scores=(1.0-alpha)*neural+alpha*lexical
        for j in range(stop-start):
            self_idx=cand_pos.get(queries[start+j]['id'])
            if self_idx is not None: scores[j,self_idx]=-np.inf
            if np.isfinite(scores[j]).sum()<7: raise ValueError(f'{queries[start+j]["id"]}: fewer than seven eligible candidates after self-exclusion.')
            idx=np.argpartition(scores[j],-7)[-7:]
            idx=idx[np.argsort(scores[j,idx])[::-1]]
            preds[queries[start+j]['id']]=[cand_ids[int(k)] for k in idx]
    del ctorch,qtorch; torch.cuda.empty_cache()
    return preds

def validate_pred(pred,queries,candidates):
    expected={q['id'] for q in queries}; allowed=set(candidates)
    if set(pred)!=expected: raise AssertionError(f'Query coverage mismatch: missing={len(expected-set(pred))}, extra={len(set(pred)-expected)}')
    for qid,ids in pred.items():
        if not isinstance(ids,list) or len(ids)!=7 or len(set(ids))!=7 or not set(ids)<=allowed:
            raise AssertionError(f'{qid}: expected exactly 7 unique IDs from its own candidate pool.')
    return True

def portal_zip(pred,name,queries,candidates):
    validate_pred(pred,queries,candidates)
    SUB_DIR.mkdir(parents=True,exist_ok=True); stage=EXP_ROOT/'submission_stage'; stage.mkdir(parents=True,exist_ok=True)
    pred_path=stage/'prediction.json'; tmp_json=stage/'prediction.json.tmp'
    tmp_json.write_text(json.dumps(pred,ensure_ascii=False,indent=2),encoding='utf-8'); os.replace(tmp_json,pred_path)
    archive=SUB_DIR/f'{name}_PORTAL_READY.zip'; tmp_zip=archive.with_suffix('.zip.tmp')
    with zipfile.ZipFile(tmp_zip,'w',compression=zipfile.ZIP_DEFLATED) as z: z.write(pred_path,'prediction.json')
    with zipfile.ZipFile(tmp_zip) as z:
        assert z.testzip() is None and z.namelist()==['prediction.json']
        roundtrip=json.loads(z.read('prediction.json'))
    validate_pred(roundtrip,queries,candidates); os.replace(tmp_zip,archive)
    print('PORTAL ZIP VALID:',archive,'| queries:',len(roundtrip),'| root member: prediction.json')
    return archive

# If a prior session completed training, restore the best validation checkpoint before inference.
if COMPLETE_MARKER.is_file():
    final_zip=SUB_DIR/'A_FullFineTune_LongRun_PORTAL_READY.zip'
    if final_zip.is_file():
        with zipfile.ZipFile(final_zip) as z:
            assert z.testzip() is None and z.namelist()==['prediction.json']
            validate_pred(json.loads(z.read('prediction.json')),TEST_Q,TEST_C)
        print('A_FullFineTune_LongRun is already complete; existing validated submission retained:',final_zip)
    else:
        if BEST_MODEL.is_file():
            best_payload=torch.load(BEST_MODEL,map_location='cpu',weights_only=False)
            if (best_payload.get('magic')!=RUN_MAGIC or best_payload.get('base_checkpoint_sha256')!=BASE_CHECKPOINT_SHA256
                or best_payload.get('dataset_fingerprint')!=DATASET_FINGERPRINT or best_payload.get('tokenizer_sha256')!=TOKENIZER_SHA256):
                raise RuntimeError('Best-model checkpoint fingerprint mismatch; refusing to create an output from stale weights.')
            retrieval_model.load_state_dict(best_payload['model_state_dict'],strict=True)
            best_model_step=int(best_payload['step']); del best_payload
        else:
            best_model_step=int(json.loads((EXP_ROOT/'training_state.json').read_text()).get('steps',-1))
            print('No periodic best checkpoint exists; evaluating the final in-memory model on full Dev.')
            fallback_best,fallback_rows=evaluate_dev_model(retrieval_model)
            if save_best_model(retrieval_model,best_model_step,fallback_best):
                best_alpha=float(fallback_best['alpha'])
        train_state=json.loads((EXP_ROOT/'training_state.json').read_text())
        if train_state['training_hours']<TOTAL_TRAIN_HOURS-0.02:
            print('Training still resumable:',train_state['training_hours'],'/',TOTAL_TRAIN_HOURS,'optimizer-hours. No final test ZIP yet.')
        else:
            # Dev alpha selection is performed on every labeled Dev query and its complete Dev candidate pool.
            print('Encoding full Dev split with selected best model, step',best_model_step)
            dev_c_emb=encode_document_cache(retrieval_model,TOKEN_CACHES['dev_c'],'best_dev_candidates',model_step=best_model_step)
            dev_q_emb=encode_document_cache(retrieval_model,TOKEN_CACHES['dev_q'],'best_dev_queries',model_step=best_model_step)
            dev_cx,dev_qx=tfidf_split(DEV_C,DEV_Q,'dev')
            dev_results=[]
            for alpha in (0.0,0.10,0.20,0.30,0.40,0.50,0.60):
                pred=rank_blend(dev_q_emb,dev_c_emb,dev_qx,dev_cx,DEV_Q,DEV_C,alpha)
                metrics=micro_metrics(pred,DEV_Q); dev_results.append({'alpha':alpha,**metrics})
                print('DEV alpha=',alpha,metrics)
            best_dev=max(dev_results,key=lambda x:x['micro_f1@7']); chosen_alpha=float(best_dev['alpha'])
            atomic_json({'model_step':best_model_step,'selected_alpha':chosen_alpha,'dev_results':dev_results,'best_dev':best_dev},EXP_ROOT/'dev_model_selection.json')
            print('Selected alpha:',chosen_alpha,'| Micro-F1@7:',best_dev['micro_f1@7'])
            # Test candidates are embedded only against their own official test candidate pool.
            test_c_emb=encode_document_cache(retrieval_model,TOKEN_CACHES['test_c'],'best_test_candidates',model_step=best_model_step)
            test_q_emb=encode_document_cache(retrieval_model,TOKEN_CACHES['test_q'],'best_test_queries',model_step=best_model_step)
            test_cx,test_qx=tfidf_split(TEST_C,TEST_Q,'test')
            test_pred=rank_blend(test_q_emb,test_c_emb,test_qx,test_cx,TEST_Q,TEST_C,chosen_alpha)
            submission=portal_zip(test_pred,'A_FullFineTune_LongRun',TEST_Q,TEST_C)
            final={'experiment':'A_FullFineTune_LongRun','architecture':'A / LegalDecoderA','method':'full_fine_tuning',
                'training_hours':train_state['training_hours'],'optimizer_steps':train_state['steps'],'best_model_step':best_model_step,
                'best_dev':best_dev,'selected_alpha':chosen_alpha,'submission':str(submission),'zip_root':['prediction.json'],
                'test_queries':len(TEST_Q),'test_candidates':len(TEST_C),'status':'validated_portal_ready'}
            atomic_json(final,COMPLETE_MARKER)
            print('FINAL RESULT:',json.dumps(final,indent=2))
else:
    train_state=json.loads((EXP_ROOT/'training_state.json').read_text()) if (EXP_ROOT/'training_state.json').is_file() else None
    if not train_state or train_state.get('training_hours',0)<TOTAL_TRAIN_HOURS-0.02:
        print('Training is paused/resumable; final Dev/Test retrieval will run after optimizer-time target is reached.')
    else:
        print('Training target complete; rerun this cell to perform final Dev/Test retrieval.')


In [ ]:
EXP_ROOT.mkdir(parents=True,exist_ok=True); SUB_DIR.mkdir(parents=True,exist_ok=True)
state_path=EXP_ROOT/'training_state.json'
training_state=json.loads(state_path.read_text()) if state_path.is_file() else None
manifest={
 'magic':RUN_MAGIC,'architecture':'A / LegalDecoderA (Phase-1 243,897,344 parameters)',
 'method':'Full fine-tuning with three-view document representation, TF-IDF hard negatives, contrastive + pairwise ranking',
 'base_checkpoint':str(BASE_CHECKPOINT),'base_checkpoint_sha256':BASE_CHECKPOINT_SHA256,
 'tokenizer_sha256':TOKENIZER_SHA256,'dataset_fingerprint':DATASET_FINGERPRINT,
 'session_safe_hours':SESSION_SAFE_HOURS,'total_optimizer_training_budget_hours':TOTAL_TRAIN_HOURS,
 'resume_checkpoint':str(LATEST_RESUME),'resume_checkpoint_exists':LATEST_RESUME.is_file(),
 'resume_checkpoint_bytes':LATEST_RESUME.stat().st_size if LATEST_RESUME.is_file() else None,
 'training_state':training_state,
 'best_model_exists':BEST_MODEL.is_file(),'best_model_bytes':BEST_MODEL.stat().st_size if BEST_MODEL.is_file() else None,
 'best_model_sha256':sha256_file(BEST_MODEL) if BEST_MODEL.is_file() else None,
 'best_metrics_exists':BEST_METRICS.is_file(),
 'final_submission_exists':(SUB_DIR/'A_FullFineTune_LongRun_PORTAL_READY.zip').is_file(),
 'timestamp':time.strftime('%Y-%m-%d %H:%M:%S'),
}
atomic_json(manifest,RUN_MANIFEST)
print(json.dumps(manifest,indent=2))
print('\nOUTPUTS TO PRESERVE:')
for p in [LATEST_RESUME,BEST_MODEL,RUN_MANIFEST,ROOT/'data_inventory.json',ROOT/'preprocessing_manifest.json',SUB_DIR/'A_FullFineTune_LongRun_PORTAL_READY.zip']:
    if p.is_file(): print(p,'| GiB=',round(p.stat().st_size/(1024**3),3))
print('Free working disk:',round(free_gb(),2),'GiB')
print('\nSAFE RESTART: Save Version -> Quick Save. In a new Kaggle session, Add Input -> Notebook Output for the previous version, then rerun all cells. The run must report a matching base-checkpoint SHA, tokenizer SHA and dataset fingerprint before optimizer resume.')


## Expected final submission

`/kaggle/working/CS781_PHASE7_ARCHA_LONGRUN/submissions/A_FullFineTune_LongRun_PORTAL_READY.zip`

The archive contains exactly one root-level `prediction.json`. Every test query maps to exactly seven unique candidate IDs from the official `test_candidates.json` pool, in descending blended retrieval score order.

## Restart procedure

Kaggle Notebook documentation currently states a maximum of 12 hours for CPU/GPU sessions and 20 GB of saved `/kaggle/working` output. The notebook stops earlier, at its safe session budget, and saves a full FP32 resume checkpoint.

1. At the safe stop, select **Save Version → Quick Save**.
2. Start a fresh session and attach the previous version using **Add Input → Notebook Output**.
3. Run all cells. The notebook restores `/kaggle/working/CS781_PHASE7_ARCHA_LONGRUN`, verifies run/checkpoint/tokenizer/dataset fingerprints, and restores optimizer, scheduler, scaler and RNG states.
4. Continue until 13.5 hours of optimizer time have accumulated. Final full-Dev model selection and Test retrieval run after training completes.

Do not upload the outer notebook-output bundle as the competition submission. Upload only `A_FullFineTune_LongRun_PORTAL_READY.zip`.

**Validation caveat:** this notebook has passed static and synthetic tests in the build environment, but the real Phase-1 checkpoint, live Kaggle T4 training throughput, peak VRAM and the complete 15–18-hour run must still be verified in Kaggle.